# 🎬 Osamah Vids — Google Colab GPU Worker for Wan2.1 Video Generation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alomriosamah1-eng/Osamah-vids/blob/main/colab/osamah_vids_wan21_worker.ipynb)

This official Google Colab notebook runs the state-of-the-art **Wan2.1 (Wan-AI/Wan2.1-T2V-1.3B-Diffusers)** and **LTX-Video** models on Google Colab's Free T4 GPU or A100 GPU, connected directly to your **Osamah Vids** web application.

### Supported Architectures:
1. **Wan2.1-T2V-1.3B**: Optimized for Colab Free Tier (T4 GPU with 16GB VRAM + CPU Offload + VAE Tiling).
2. **Wan2.1-T2V-14B**: Studio Grade 720p/1080p generation (A100 GPU).
3. **LTX-Video (2B DiT)**: Ultra-fast real-time DiT video generation.

### ⚡ الخطوة 1: التحقق من كرت الشاشة (GPU Verification)

In [ ]:
!nvidia-smi

### 📦 الخطوة 2: تثبيت حزم ونماذج الذكاء الاصطناعي (Install AI Packages)

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate torch torchvision torchaudio sentencepiece imageio imageio-ffmpeg fastapi uvicorn pydantic pyngrok

### 🚀 الخطوة 3: تشغيل خادم المعالجة واستخراج رابط النفق السحابي (Run Worker & Expose Public URL)

In [ ]:
import os
import time
import subprocess

# Clone or pull latest Osamah Vids repository
!rm -rf Osamah-vids 2>/dev/null || true
!git clone https://github.com/alomriosamah1-eng/Osamah-vids.git
%cd Osamah-vids/colab

# Start the FastAPI worker in background
worker_proc = subprocess.Popen(['python3', 'worker.py'])
time.sleep(3)

print("\n=======================================================")
print("✅ Osamah Vids Worker is RUNNING on port 8000!")
print("Copy the public URL below and paste it in Osamah Vids UI Settings:")
print("=======================================================\n")

# Expose port 8000 with localtunnel
!npx -y localtunnel --port 8000

### 🧪 الخطوة 4 (اختياري): اختبار توليد فيديو محلياً داخل كولاب مباشرة (Standalone Test)

In [ ]:
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video

print("Loading Wan2.1 model for direct Colab test...")
pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
if torch.cuda.is_available():
    pipe.enable_model_cpu_offload()

prompt = "A cinematic shot of a glowing futuristic sports car driving through a neon cyber city at night, 4k ultra realistic, highly detailed, dramatic lighting"
video = pipe(prompt=prompt, height=480, width=832, num_frames=45, num_inference_steps=30).frames[0]
export_to_video(video, "test_output.mp4", fps=16)
print("✅ Test video generated and saved as test_output.mp4!")